# Predykcja Market Value - Fortune 500

Cel: przewidziec wartosc rynkowa (Market Value) firm z listy Fortune 500 na podstawie danych finansowych.

Dane: https://www.kaggle.com/datasets/mirzayasirabdullah07/fortune-500-companies-us/data

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt

from src.scripts.eda import run_eda
from src.preprocessing.preprocessing import run_preprocessing
from src.modelling.models import run_modelling
from src.modelling.optimization import run_optimization
from src.scripts.interpretacja import run_interpretation

## 1. Wczytanie i analiza danych (EDA)

In [ ]:
df = run_eda()
df.head()

In [ ]:
from IPython.display import Image, display
for fname in ["histogramy", "macierz_korelacji", "scatter_plots", "boxploty"]:
    display(Image(f"wyniki/{fname}.png"))

**Wnioski z histogramow:**
- Wszystkie zmienne finansowe maja silny skos prawy - wiekszosc firm skupiona w dolnym zakresie wartosci, najwieksze (Walmart, Apple) mocno odstaja
- Skosnosc Market Value = 4.68 - bardzo skosny rozklad, potrzerbna transformacja logarytmiczna 
- Liczba pracownikow ma ekstremalny rozklad - kilka firm zatrudnia ponad milion osob

**Wnioski ze scatter plotow:**
- Profits vs Market Value - najbardziej liniowa zaleznosc, potwierdza wynik z macierzy korelacji
- Widac outlinery: Walmart ma ogromne przychody ale wycena nie jest proporcjonalnie wysoka
- Firmy tech (Apple, Alphabet) maja wysoka wycene przy umiarkowanych przychodach
- Zaleznosc Assets vs MV jest bardzo slaba i rozproszona

**Wnioski z boxplotow:**
- Market Value ma 43 outlinery. Sa to firmy technologiczne i finansowe z bardzo wysoka wycena
- Outlinery to prawdziwe dane duzych firm, nie sa to bledy pomiarowe

## 2. Preprocessing i Feature Engineering

In [ ]:
X_train, X_test, y_train, y_test, scaler, features = run_preprocessing(df)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")
print(f"Cechy ({len(features)}): {features}")

## 3. Modelowanie

# Metryki oceny modelu:
- MAE sredni blad bezwzgledny; kazdy blad liczy sie tak samo; latwy do interpretacji
- RMSE  karze duze bledy nieproporcjonalnie (przez kwadrat); wrazliwy na wartosci odstajace
- R2  ile wariancji zmiennej docelowej wyjasnia model (0 = bezuzyteczny, 1 = idealny)
- Uzywamy wszystkich trzech bo razem daja pelniejszy obraz jakosci modelu

In [ ]:
results, models, predictions = run_modelling(X_train, X_test, y_train, y_test)

In [ ]:
from IPython.display import Image, display
for fname in ["porownanie_modeli", "predykcje_vs_rzeczywiste"]:
    display(Image(f"wyniki/{fname}.png"))

**Wnioski z modelowania:**
- Linear Regression daje najwyzsze R2 na zbiorze testowym- przy malej ilosci danych (464 prob) prostszy model moze generalizowac lepiej
- Wszystkie modele maja duzy max blad (ok. 365 tys mln $) - dotycza firm tech z wysoka premia rynkowa, ktorych wartosc trudno przewidziec z samych danych fin.
- Punkty blisko linii przerywanej = dobre predykcje, duze odchylenia = firmy trudne do wyceny
- R2 okolo 0.5 oznacza ze model wyjasnia ok. 50% zmiennosci Market Value - reszta to czynniki nieuwzglednione w danych (marka, innowacje, oczekiwania rynku)

## 4. Optymalizacja hiperparametrow

In [ ]:
best_model, best_name, results_opt, study_rf, study_xgb, study_lasso = run_optimization(
    X_train, X_test, y_train, y_test, results, n_trials=30
)

In [ ]:
from IPython.display import Image, display
for fname in ["optymalizacja_porownanie", "optuna_parametry"]:
    display(Image(f"wyniki/{fname}.png"))

In [ ]:
from src.modelling.optimization import _plot_history

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
_plot_history(study_rf,    axes[0], "RF - Historia optymalizacji")
_plot_history(study_xgb,   axes[1], "XGB - Historia optymalizacji")
_plot_history(study_lasso, axes[2], "Lasso - Historia optymalizacji")
plt.suptitle("Optuna: poprawa R2 w kolejnych probach", fontsize=13)
plt.tight_layout()
plt.show()

**Wnioski z optymalizacji (Optuna):**
- Optuna uzywa algorytmu TPE (Tree-structured Parzen Estimator) - w odroznieniu od GridSearchCV nie przeszukuje siatki, ale buduje model probabilistyczny i kieruje kolejne proby w obszary o wyzszym R2
- Przestrzen przeszukiwan jest szersza i ciagla: np. `learning_rate` probkowane logarytmicznie miedzy 0.01 a 0.3 zamiast 3 z gory ustalonych wartosci
- 30 prob Optuna vs 36 kombinacji GridSearch dla RF - podobna liczba fitow, ale Optuna pokrywa wiekszy obszar i adaptatywnie skupia sie na obiecujacych regionach
- `optuna.logging.set_verbosity(WARNING)` wycisza verbose output; mozna uzyc `optuna.visualization` do wykresu historii optymalizacji

## 5. Interpretacja wynikow

In [ ]:
importance = run_interpretation(best_model, X_test, features)

In [ ]:
from IPython.display import Image, display
for fname in ["feature_importance", "shap_summary", "shap_bar"]:
    display(Image(f"wyniki/{fname}.png"))

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

# PDP dla 4 najwazniejszych cech
paired_imp = list(zip(features, importance))
paired_imp.sort(key=lambda x: x[1], reverse=True)
top_4 = [feat for feat, _ in paired_imp[:4]]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
ax_list = [axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]]
for feat, ax in zip(top_4, ax_list):
    idx = features.index(feat)
    PartialDependenceDisplay.from_estimator(
        best_model, X_test, features=[idx],
        feature_names=features, ax=ax, kind="average"
    )
    ax.set_title(f"PDP: {feat}")

plt.suptitle("Partial Dependence Plots", fontsize=14)
plt.tight_layout()
plt.show()

**Wnioski z Feature Importance:**
- Profits zdecydowanie dominuje (~73% waznosci)- zyski sa najwazniejszym czynnikiem predykcji wartosci rynkowej
- Number of Employees na drugim miejscu, czyli  wielkosc firmy ma znaczenie
- Revenue Change (dynamika wzrostu) tez istotne- rynek wycenia firmy rosnace wyzej
- Log_Revenues i Log_Assets maja zerowa waznosc - model juz korzysta z oryginalnych wartosci, transformacje log nie daja dodatkowej informacji


**Wnioski z analizy SHAP:**
- SHAP potwierdza dominacje Profits- wysokie zyski (czerwone punkty po prawej) zdecydowanie podnoszaja predykcje Market Value
- W odroznieniu od Feature Importance, SHAP pokazuje tez KIERUNEK wplywu- nie tylko ktora cecha jest wazna, ale jak wplywa
- Niskie zyski (niebieskie punkty po lewej) obnizaja predykcje- zaleznosc jest symetryczna
- Rozrzut punktow przy Number of Employees wskazuje na nieliniowy wplyw- duza firma to nie zawsze wysoka wycena

**Wnioski z PDP:**
- Profits: wyrazna rosnaca krzywa - im wyzsze zyski, tym wyzsza predykcja Market Value. Efekt jest silny i monotoniczny
- Number of Employees: plaska krzywa z naglym wzrostem dla bardzo duzych firm - wielkosc zatrudnienia ma znaczenie dopiero przy gigantach
- Revenue Change: firmy z szybszym wzrostem przychodow sa wyceniane wyzej
- Nieliniowe ksztalty krzywych potwierdzaja, ze modele drzewiastee (RF, XGBoost) lepiej chwytaja te zaleznosci niz regresja liniowa

## Podsumowanie

- **Najwazniejsza cecha**: Profits (zyski) - dominuje zarowno w Feature Importance jak i SHAP
- **Najlepszy model**: wyznaczony dynamicznie powyzej przez Optuna
- **R2 ~0.52**: model wyjasnia ok. polowe zmiennosci Market Value. Reszta to czynniki nieobecne w danych (marka, patenty, oczekiwania rynku)
- **Glowne ograniczenie**: maly dataset (464 firmy) i brak cech jakosciowych (branza, innowacyjnosc) - przy wiekszej ilosci danych i bogatszych cechach wyniki bylyby lepsze

In [ ]:
print(f"Najlepszy model: {best_name}")
print("Wykresy zapisane w: wyniki/")